# Dante — benchmark Qwen versus Mamba no Ollama

Este notebook reproduz o benchmark local do projeto usando **Ollama em uma GPU do Google Colab**. Ele mede os mesmos 20 prompts contra `qwen2.5:3b` e `hf.co/bartowski/mamba-2.8b-hf-GGUF:Q4_K_M`, mas usa contexto e limite de saída maiores.

Antes de executar: em **Runtime > Change runtime type**, selecione uma GPU. O Ollama é obrigatório neste notebook; não há respostas simuladas. A primeira execução inclui o carregamento dos modelos, então compare também as métricas após o aquecimento.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version

## Instalar e iniciar o Ollama

In [ ]:
!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install ollama pandas

In [ ]:
import os
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama-colab.log", "w"),
    stderr=subprocess.STDOUT,
)
for _ in range(30):
    result = subprocess.run(["curl", "-sf", "http://127.0.0.1:11434/api/tags"], capture_output=True)
    if result.returncode == 0:
        print("Ollama pronto.")
        break
    time.sleep(1)
else:
    raise RuntimeError("Ollama não iniciou. Consulte /tmp/ollama-colab.log")

print(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip())

## Baixar os mesmos modelos

O download pode levar alguns minutos e consome aproximadamente 4 GB. O modelo Mamba é o mesmo usado no benchmark local.

In [ ]:
!ollama pull qwen2.5:3b
!ollama pull hf.co/bartowski/mamba-2.8b-hf-GGUF:Q4_K_M
!ollama list

## Benchmark

A configuração usa `num_ctx=8192`, `num_predict=256` e temperatura `0.1`. Cada modelo é aquecido uma vez antes das medições. Os campos `load_duration`, `prompt_eval_duration` e `eval_duration` vêm da resposta do Ollama e permitem separar carregamento de inferência.

In [ ]:
import json
import platform
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path

import ollama

MODELS = {
    "qwen": "qwen2.5:3b",
    "mamba": "hf.co/bartowski/mamba-2.8b-hf-GGUF:Q4_K_M",
}
PROMPTS = [
    "Explique em duas frases por que a observação contínua ajuda uma IA a lembrar contexto.",
    "Descreva uma rotina de trabalho eficiente para um time de engenharia de IA.",
    "Escreva um resumo curto de um artigo sobre memória de longo prazo em modelos de linguagem.",
    "Dê um exemplo de prompt útil para avaliar coerência em respostas curtas.",
    "Explique o que é isolamento do modelo em um sistema multi-agente.",
    "Crie uma frase criativa sobre uma estação do ano e um editor de código.",
    "Como você descreveria a diferença entre atenção e estado recorrente em LLMs?",
    "Escreva um texto técnico sobre embeddings e recuperação por similaridade.",
    "Responda em português em três frases sobre um cenário de automação doméstica.",
    "Explique por que benchmarks devem incluir latência e qualidade em conjunto.",
    "Crie uma mensagem curta para um usuário que acabou de começar a usar um assistente local.",
    "Descreva em 2 linhas o trade-off entre velocidade e profundidade de raciocínio.",
    "Quais cuidados um sistema de observação contínua deve ter com privacidade?",
    "Escreva uma pergunta útil para avaliar a capacidade de reflexão de uma IA.",
    "Descreva uma experiência de uso de um diário humano em um sistema de IA.",
    "Explique em poucas palavras o que é um state space model.",
    "Crie uma resposta curta para uma tela de dashboard de produtividade.",
    "Escreva uma observação reflexiva sobre rotina, atenção e memória.",
    "Explique por que um fallback é importante em um pipeline de modelos locais.",
    "Resuma em duas frases como o contexto pode melhorar respostas de IA.",
]
OPTIONS = {"num_ctx": 8192, "num_predict": 256, "temperature": 0.1}
client = ollama.Client(host="http://127.0.0.1:11434", timeout=120)

def generate(model_name, prompt):
    started = time.perf_counter()
    response = client.generate(
        model=model_name,
        prompt=prompt,
        options=OPTIONS,
        keep_alive="10m",
    )
    wall_seconds = time.perf_counter() - started
    text = response.response if hasattr(response, "response") else response.get("response", "")
    return {
        "response": text.strip(),
        "wall_seconds": wall_seconds,
        "prompt_tokens": getattr(response, "prompt_eval_count", None),
        "response_tokens": getattr(response, "eval_count", None),
        "load_seconds": (getattr(response, "load_duration", 0) or 0) / 1e9,
        "prompt_eval_seconds": (getattr(response, "prompt_eval_duration", 0) or 0) / 1e9,
        "eval_seconds": (getattr(response, "eval_duration", 0) or 0) / 1e9,
    }

def summarize(rows):
    valid = [row for row in rows if row["response"]]
    walls = [row["wall_seconds"] for row in valid]
    response_tokens = [row["response_tokens"] for row in valid if row["response_tokens"]]
    eval_seconds = [row["eval_seconds"] for row in valid if row["eval_seconds"]]
    total_tokens = sum(response_tokens)
    total_eval = sum(eval_seconds)
    return {
        "valid": len(valid),
        "total": len(rows),
        "wall_p50_seconds": statistics.median(walls) if walls else None,
        "wall_p95_seconds": sorted(walls)[min(len(walls) - 1, int(len(walls) * 0.95))] if walls else None,
        "avg_response_tokens": statistics.mean(response_tokens) if response_tokens else None,
        "ollama_eval_tokens_per_second": total_tokens / total_eval if total_eval else None,
    }

results = {}
for label, model_name in MODELS.items():
    print(f"Aquecendo {label}: {model_name}")
    generate(model_name, "Responda apenas: WARMUP_OK")
    rows = []
    for index, prompt in enumerate(PROMPTS, 1):
        print(f"{label}: {index}/{len(PROMPTS)}", end="\r")
        try:
            row = generate(model_name, prompt)
            row.update({"prompt": prompt, "index": index, "error": None})
        except Exception as exc:
            row = {"prompt": prompt, "index": index, "response": "", "error": repr(exc)}
        rows.append(row)
    results[label] = {"model": model_name, "summary": summarize(rows), "results": rows}
    print(f"{label}: concluído")

payload = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "environment": {
        "platform": platform.platform(),
        "python": platform.python_version(),
        "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip(),
        "options": OPTIONS,
        "ollama_only": True,
    },
    "systems": results,
}
output_path = Path("/content/compare_colab_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + ".json")
output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(output_path)


In [ ]:
import pandas as pd

summary = []
for label, data in payload["systems"].items():
    row = {"modelo": label, **data["summary"]}
    summary.append(row)
pd.DataFrame(summary)

## Comparar com o benchmark local

Faça upload de `benchmarks/results/compare_20261108.json` usando a célula abaixo. A comparação é apenas indicativa: o notebook usa `num_ctx=8192` e `num_predict=256`, enquanto o benchmark local de 08/11 usou geração limitada a 128 tokens.

In [ ]:
from google.colab import files

uploaded = files.upload()
for filename in uploaded:
    local_payload = json.loads(Path(filename).read_text(encoding="utf-8"))
    comparison = []
    for label, data in payload["systems"].items():
        local_summary = local_payload["systems"].get(label, {}).get("summary", {})
        comparison.append({
            "modelo": label,
            "colab_p50_s": data["summary"]["wall_p50_seconds"],
            "local_p50_s": (local_summary.get("p50_ms") or 0) / 1000,
            "colab_tokens_s": data["summary"]["ollama_eval_tokens_per_second"],
            "local_tokens_s": local_summary.get("throughput_tokens_per_s"),
        })
    display(pd.DataFrame(comparison))


In [ ]:
from google.colab import files
files.download(str(output_path))